#### Loading cleaned datasets

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ECB API data
df = pd.read_csv('inflation_clean.csv')

# Eurostat API
df2 = pd.read_csv('interest_clean.csv')


df.head(25)

,date,country,inflation
0,2015-01,DE,-0.5
1,2015-02,DE,-0.2
2,2015-03,DE,0.3
3,2015-04,DE,1.0
4,2015-05,DE,1.6
5,2015-06,DE,1.1
6,2015-07,DE,1.3
7,2015-08,DE,1.1
8,2015-09,DE,0.8
9,2015-10,DE,1.2


In [2]:
df2.head()

,date,interest
0,2015-12,-0.30
1,2016-03,-0.40
2,2019-09,-0.50
3,2022-07,0.00
4,2022-09,0.75


---

In [3]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df2["date"] = pd.to_datetime(df2["date"], errors="coerce")

In [4]:
df2["interest"] = pd.to_numeric(df2["interest"], errors="coerce")

# Keep only what we need
df2 = df2[["date", "interest"]]

# Remove the duplicates
df2 = df2.drop_duplicates(subset=["date"])
df = df.drop_duplicates(subset=["country", "date"])



### Merging the datasets via date

In [5]:
# Making sure the date format matches each dataset
df['date'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
df2['date'] = pd.to_datetime(df2['date']).dt.to_period('M').astype(str)

In [6]:
# Merge the two datsets
df_merged = pd.merge(df,df2, on='date', how='left')

# Sort the values
df_merged = df_merged.sort_values(['country', 'date'])

In [7]:
df_merged

,date,country,inflation,interest
0,2015-01,DE,-0.5,NaN
1,2015-02,DE,-0.2,NaN
2,2015-03,DE,0.3,NaN
3,2015-04,DE,1.0,NaN
4,2015-05,DE,1.6,NaN
...,...,...,...,...
427,2023-08,IT,5.5,3.75
428,2023-09,IT,5.6,4.00
429,2023-10,IT,1.8,NaN
430,2023-11,IT,0.6,NaN


In [8]:
# Filling in the Nan values
df_merged = df_merged.sort_values(["country", "date"])

df_merged["interest"] = df_merged["interest"].ffill().bfill()

In [9]:
df_merged.tail()

,date,country,inflation,interest
427,2023-08,IT,5.5,3.75
428,2023-09,IT,5.6,4.00
429,2023-10,IT,1.8,4.00
430,2023-11,IT,0.6,4.00
431,2023-12,IT,0.5,4.00


In [10]:
df_merged.shape

(432, 4)

In [14]:
df_merged['country'].unique()

array(['DE', 'ES', 'FR', 'IT'], dtype=object)

In [11]:
print(df_merged[df_merged["inflation"].isna()].head(20))

Empty DataFrame
Columns: [date, country, inflation, interest]
Index: []


In [12]:
print(df_merged[df_merged["inflation"].isna()]["date"].unique())

[]


In [13]:
df_merged.head(50)

,date,country,inflation,interest
0,2015-01,DE,-0.5,-0.3
1,2015-02,DE,-0.2,-0.3
2,2015-03,DE,0.3,-0.3
3,2015-04,DE,1.0,-0.3
4,2015-05,DE,1.6,-0.3
5,2015-06,DE,1.1,-0.3
6,2015-07,DE,1.3,-0.3
7,2015-08,DE,1.1,-0.3
8,2015-09,DE,0.8,-0.3
9,2015-10,DE,1.2,-0.3
